In [1]:

import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import TensorDataset
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

from sklearn.metrics import classification_report


https://www.kaggle.com/datasets/ziya07/network-traffic-anomaly-detection-dataset/data

## Load the Data

In [2]:
df = pd.read_csv('./embedded_system_network_security_dataset.csv')
df.head()

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.0,0.834066,0.534891,0.0,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.0,0.147196,0.990757,0.0,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.0,0.855192,0.031781,0.0,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.0,0.153220,0.169958,0.0,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.0,0.923916,0.552053,0.0,True,False,False,True,False,False,False,True,False


In [3]:
df.isna().sum()

packet_size              0
inter_arrival_time       0
src_port                 0
dst_port                 0
packet_count_5s          0
mean_packet_size         0
spectral_entropy         0
frequency_band_energy    0
label                    0
protocol_type_TCP        0
protocol_type_UDP        0
src_ip_192.168.1.2       0
src_ip_192.168.1.3       0
dst_ip_192.168.1.5       0
dst_ip_192.168.1.6       0
tcp_flags_FIN            0
tcp_flags_SYN            0
tcp_flags_SYN-ACK        0
dtype: int64

## Feature removal and DPT

In [4]:
df.drop(['src_ip_192.168.1.2', 'src_ip_192.168.1.3', 'dst_ip_192.168.1.5', 'dst_ip_192.168.1.6'], axis=1, inplace=True)

In [5]:
df['protocol_type_TCP'] = df['protocol_type_TCP'].astype(int)
df['protocol_type_UDP'] = df['protocol_type_UDP'].astype(int)
df['tcp_flags_FIN'] = df['tcp_flags_FIN'].astype(int)
df['tcp_flags_SYN'] = df['tcp_flags_SYN'].astype(int)
df['tcp_flags_SYN-ACK'] = df['tcp_flags_SYN-ACK'].astype(int)

In [6]:
df['src_port'] = df['src_port'] / 65535
df['dst_port'] = df['dst_port'] / 65535

In [7]:
df['label'].value_counts()

label
0.0    900
1.0    100
Name: count, dtype: int64

In [8]:

#smote = SMOTE(sampling_strategy=1.0, random_state=42)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
#X_train, y_train = smote.fit_resample(train_df.drop(['label'], axis=1), train_df['label'])
X_train, y_train = train_df.drop(['label'], axis=1), train_df['label']


In [9]:
y_train.value_counts()

label
0.0    720
1.0     80
Name: count, dtype: int64

In [10]:
X_train.shape, y_train.shape

((800, 13), (800,))

In [11]:
dataset_train = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.long))
dataset_test = TensorDataset(torch.tensor(test_df.drop(['label'], axis=1).values, dtype=torch.float32), torch.tensor(test_df['label'].values, dtype=torch.long))

## Define the Neural Network

In [15]:
class MyNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(13, 20)       
        self.activation_1 = nn.ReLU()          # Define ReLU activation function
        self.layer_2 = nn.Linear(20, 20)        
        self.activation_2 = nn.ReLU() 
        self.layer_3 = nn.Linear(20, 20)        
        self.activation_3 = nn.ReLU() # Define Softmax activation function on last dimension
        self.layer_4 = nn.Linear(20, 20)        # Define dense layer with an input dimension of 20 and an output dimension of 2
        self.activation_4 = nn.ReLU() # Define Softmax activation function on last dimension
        self.layer_5 = nn.Linear(20, 2)        # Define dense layer with an input dimension of 20 and an output dimension of 2
        self.activation_5 = nn.Sigmoid() # Define Softmax activation function on last dimension 

    def forward(self, X):
        X = self.layer_1(X)                    # Pass data through layer 1
        X = self.activation_1(X)                 # Pass data through activation function 1
        X = self.layer_2(X)                    # Pass data through layer 2
        X = self.activation_2(X)                 # Pass data through activation function 2
        X = self.layer_3(X)                    # Pass data through layer 3
        X = self.activation_3(X)                 # Pass data through activation function 3
        X = self.layer_4(X)                    # Pass data through layer 4
        X = self.activation_4(X)                 # Pass data through activation function 4
        X = self.layer_5(X)                    # Pass data through layer 5
        X = self.activation_5(X)                 # Pass data through activation function 5
        return X
    
model = MyNetwork()
print(model)

MyNetwork(
  (layer_1): Linear(in_features=13, out_features=20, bias=True)
  (activation_1): ReLU()
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (activation_2): ReLU()
  (layer_3): Linear(in_features=20, out_features=20, bias=True)
  (activation_3): ReLU()
  (layer_4): Linear(in_features=20, out_features=20, bias=True)
  (activation_4): ReLU()
  (layer_5): Linear(in_features=20, out_features=2, bias=True)
  (activation_5): Sigmoid()
)


## Train the model

In [18]:
criterion = CrossEntropyLoss(weight=torch.tensor([1.0, 9.0]))
# We use the Adam optimizer on the parameters of our network and define the learning rate (other parameters may be defined as well)
optimizer = Adam(model.parameters(), lr=0.001)

# Define how we train during an epoch
def train_single_epoch(network: nn.Module, dataset: TensorDataset) -> nn.Module:
    # Define a data loader, allowing us to shuffle data and provide a batch size
    data = DataLoader(dataset, batch_size=32, shuffle=True)
    # Add a progress bar around dataset (optional)
    data = tqdm(data, desc='Training')

    # Loop over data in dataset
    for X_train, y_true in data:

        # Clear any remaining tracking of gradients for backpropagation
        optimizer.zero_grad()
        # Do forward propagation by making a prediction
        y_pred = network(X_train)
        # Compute loss between predicted output and actual output
        loss = criterion(y_pred, y_true)
        # Backpropagate loss
        loss.backward()
        # Let the optimizer take a step
        optimizer.step()

    # Return the updated network
    return network

# We can train for multiple epochs
for epoch in range(10):
    model = train_single_epoch(model, dataset_train)

Training: 100%|██████████| 25/25 [00:00<00:00, 534.92it/s]


## Test the model

In [19]:
from sklearn.metrics import classification_report

# Get X_test and y_test from dest data
X_test, y_test = dataset_test.tensors
# Perform prediction
y_pred = model(X_test)
# Get highest index from prediction (to match with expected label)
y_pred = y_pred.argmax(dim=-1)

print(classification_report(
    y_true = y_test.numpy(),
    y_pred = y_pred.numpy(),
    digits = 4,
))

              precision    recall  f1-score   support

           0     0.9291    0.6556    0.7687       180
           1     0.1507    0.5500    0.2366        20

    accuracy                         0.6450       200
   macro avg     0.5399    0.6028    0.5026       200
weighted avg     0.8513    0.6450    0.7155       200

